In [ ]:
# ============================================================
# Week 11 BBO Capstone Script
# Theme: Clustering-Based Optimisation
# Uses 20 data points
# GP + SVM + MLP + CNN + KMeans Cluster Guidance
# ============================================================

import numpy as np
import pandas as pd
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances

import torch
import torch.nn as nn
import torch.optim as optim

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

# ============================================================
# Historical Inputs: Weeks 1–10
# ============================================================

historical_inputs = [
    [
        [0.761024, 0.713000],
        [0.732637, 0.906564],
        [0.522581, 0.591593, 0.350176],
        [0.564020, 0.473834, 0.390972, 0.258427],
        [0.196777, 0.892275, 0.855813, 0.891829],
        [0.728785, 0.146949, 0.767044, 0.739699, 0.020789],
        [0.016700, 0.532618, 0.280647, 0.222769, 0.407487, 0.748018],
        [0.035844, 0.064098, 0.010711, 0.024463, 0.361511, 0.783344, 0.515157, 0.880442],
    ],
    [
        [0.761024, 0.713000],
        [0.732637, 0.906564],
        [0.522581, 0.591593, 0.350176],
        [0.564020, 0.473834, 0.390972, 0.258427],
        [0.196777, 0.892275, 0.855813, 0.891829],
        [0.728785, 0.146949, 0.767044, 0.739699, 0.020789],
        [0.016700, 0.532618, 0.280647, 0.222769, 0.407487, 0.748018],
        [0.035844, 0.064098, 0.010711, 0.024463, 0.361511, 0.783344, 0.515157, 0.880442],
    ],
    [
        [0.751825, 0.811946],
        [0.980817, 0.626715],
        [0.996446, 0.737055, 0.850924],
        [0.404477, 0.413254, 0.303108, 0.434359],
        [0.521278, 0.505138, 0.985473, 0.994545],
        [0.341155, 0.021278, 0.626487, 0.970661, 0.032762],
        [0.431390, 0.173879, 0.071339, 0.124473, 0.403592, 0.624180],
        [0.064214, 0.412793, 0.081589, 0.008195, 0.974438, 0.216196, 0.139173, 0.110624],
    ],
    [
        [0.702630, 0.955980],
        [0.700329, 1.000000],
        [0.977206, 0.330727, 0.000017],
        [0.393932, 0.393230, 0.377559, 0.426151],
        [0.449325, 0.603715, 1.000000, 1.000000],
        [0.941155, 0.654101, 0.079248, 0.961137, 0.229746],
        [0.153100, 0.236737, 0.190018, 0.000000, 0.365128, 0.744155],
        [0.103762, 0.017606, 0.204751, 0.194101, 0.745367, 0.674182, 0.125362, 0.044024],
    ],
    [
        [0.012715, 0.992415],
        [0.026867, 0.999598],
        [0.955194, 0.008179, 0.700062],
        [0.425212, 0.377719, 0.427527, 0.420346],
        [0.479838, 0.890814, 1.000000, 1.000000],
        [0.137088, 0.268975, 0.283122, 0.986752, 0.045377],
        [0.000000, 0.450181, 0.211028, 0.000000, 0.353842, 1.000000],
        [0.000000, 0.000000, 0.301242, 0.216500, 0.000000, 1.000000, 0.000000, 1.000000],
    ],
    [
        [0.012658, 0.001204],
        [0.803770, 0.997320],
        [0.034249, 0.018981, 0.451567],
        [0.485961, 0.345148, 0.465576, 0.429334],
        [0.801922, 1.000000, 1.000000, 1.000000],
        [0.586987, 0.182795, 0.646577, 0.709830, 0.091668],
        [0.251627, 0.008515, 0.005607, 0.024728, 0.411581, 0.615802],
        [0.103911, 0.000000, 0.045943, 0.011461, 0.877426, 0.770540, 0.000000, 0.000000],
    ],
    [
        [0.058297, 0.998421],
        [0.854811, 0.022906],
        [0.507856, 0.945814, 0.000000],
        [0.425212, 0.377719, 0.427527, 0.420346],
        [1.000000, 1.000000, 1.000000, 1.000000],
        [0.548689, 0.090612, 0.596270, 0.738400, 0.042076],
        [0.446362, 0.191624, 0.030942, 0.129517, 0.412839, 0.618823],
        [0.123636, 0.877898, 0.029459, 0.738588, 0.965005, 0.127021, 0.159880, 0.078839],
    ],
    [
        [0.343980, 0.277258],
        [0.743771, 0.999666],
        [0.532769, 0.510027, 0.622979],
        [0.415714, 0.468961, 0.413078, 0.401230],
        [0.999088, 0.993081, 1.000000, 1.000000],
        [0.579004, 0.125422, 0.692500, 0.577310, 0.083764],
        [0.405202, 0.209702, 0.139382, 0.083729, 0.360804, 0.642496],
        [0.040705, 0.000000, 0.156112, 0.017086, 0.924711, 0.521550, 0.000000, 0.000000],
    ],
    [
        [0.354005, 0.996726],
        [0.700273, 1.000000],
        [0.377858, 0.585068, 0.091651],
        [0.420631, 0.428434, 0.390046, 0.387683],
        [0.996557, 1.000000, 1.000000, 1.000000],
        [0.550145, 0.147473, 0.642053, 0.698181, 0.057541],
        [0.403786, 0.202905, 0.140516, 0.064670, 0.321231, 0.668381],
        [0.000000, 0.072834, 0.000000, 0.000000, 0.464965, 0.337506, 0.087765, 0.000000],
    ],
    # Week 10 latest submitted inputs
    [
        [0.206165, 0.991956],
        [0.700269, 1.000000],
        [0.645025, 0.608525, 0.476381],
        [0.420631, 0.428434, 0.390046, 0.387683],
        [1.000000, 0.998063, 1.000000, 1.000000],
        [0.558987, 0.158160, 0.653905, 0.696695, 0.048854],
        [0.401880, 0.160448, 0.133587, 0.058632, 0.301206, 0.662879],
        [0.004318, 0.278304, 0.008528, 0.973578, 0.995037, 0.716242, 0.074606, 0.124800],
    ],
]

historical_outputs = [
    [-7.565331332744927e-18, 0.38412771439907634, -0.04757757182509677, -4.83054096204485, 1257.680268889983, -0.7671825918815833, 1.2394822144938658, 9.5430069620611],
    [-7.565331332744927e-18, 0.5530064492925906, -0.03333218343962805, -4.83054096204485, 1257.680268889983, -0.8072367077314392, 1.2394822144938658, 9.5430069620611],
    [2.495129202697582e-35, 0.06646691679004207, -0.04126707799435369, -0.7158150747637886, 1769.2992166742467, -0.721361811601727, 1.493591747104962, 9.7741312776374],
    [-8.189634555426656e-79, 0.5980717829505922, -0.12802055717541724, 0.2735224699683667, 2027.715300336871, -1.6878746934171067, 1.120167075371798, 9.8898511444579],
    [0.0, 0.008971860858651539, -0.18597539887299502, 0.5994256847352308, 3420.1124772143226, -1.0804901380041434, 0.3185961294770624, 9.199106282308],
    [5.086481560089295e-240, 0.16067534592584287, -0.08745083083260485, -1.4671093305843352, 6043.928918914933, -0.40067561435372423, 0.926482363950363, 9.744890331552],
    [7.9e-322, 0.16772685949703448, -0.13355731033976717, 0.5994256847352308, 8662.4825, -0.608209779199786, 1.3704234693682886, 8.9091791017714],
    [-2.0819799102444868e-19, 0.44330426203632317, -0.07042964248553193, -0.6289225223854156, 8512.800341126238, -0.5754696457088976, 1.5748123144599588, 9.8265157456615],
    [4.2540685116738334e-149, 0.6219708025012802, -0.06416267878508933, 0.4333470070682819, 8596.241138414929, -0.4408325725903705, 1.5374943941066745, 9.7571234923455],
    [5.325469814333144e-217, 0.5903771601872436, -0.011085004696869778, 0.4333470070682819, 8625.101349506369, -0.5146778048598337, 1.512982623511204, 9.1228717981795],
]

# ============================================================
# Helper Functions
# ============================================================

def pair_exists(X, y, point, output):
    same_x = np.all(np.isclose(X, point.reshape(1, -1), atol=1e-10), axis=1)
    same_y = np.isclose(y, output, atol=1e-10)
    return np.any(same_x & same_y)

def normalize_score(values):
    values = np.array(values, dtype=float)
    std = np.std(values)
    if std < 1e-12:
        return np.zeros_like(values)
    return (values - np.mean(values)) / std

def adaptive_strategy_params(dim, latest_improved, plateau_detected):
    if dim <= 3:
        beta = 1.2 if latest_improved else 1.9
        noise = 0.025 if latest_improved else 0.050
        temperature = 0.75 if latest_improved else 1.15
    elif dim <= 5:
        beta = 1.6 if latest_improved else 2.5
        noise = 0.040 if latest_improved else 0.075
        temperature = 0.85 if latest_improved else 1.25
    else:
        beta = 2.1 if latest_improved else 3.1
        noise = 0.070 if latest_improved else 0.110
        temperature = 0.95 if latest_improved else 1.35

    if plateau_detected:
        beta *= 1.20
        noise *= 1.25
        temperature *= 1.15

    return beta, noise, temperature

def cluster_reasoning(best_cluster, cluster_mean, boundary_distance, nearest_centroid_distance):
    return (
        f"Query targets cluster {best_cluster}, which has a strong mean output of {cluster_mean:.6f}. "
        f"The candidate is guided by centroid proximity ({nearest_centroid_distance:.6f}) "
        f"and boundary distance ({boundary_distance:.6f}), balancing local cluster refinement with controlled exploration."
    )

# ============================================================
# CNN Surrogate
# ============================================================

class CNN1DSurrogate(nn.Module):
    def __init__(self, input_dim, channels=16):
        super().__init__()
        self.conv1 = nn.Conv1d(1, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(channels, channels * 2, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear((channels * 2) * input_dim, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x).squeeze(-1)

def train_cnn(X, y, channels=16, epochs=650, lr=0.008):
    X_tensor = torch.tensor(X, dtype=torch.float32)

    y_mean = np.mean(y)
    y_std = np.std(y) if np.std(y) > 1e-12 else 1.0

    y_scaled = (y - y_mean) / y_std
    y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

    model = CNN1DSurrogate(X.shape[1], channels=channels)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    for _ in range(epochs):
        optimizer.zero_grad()
        pred = model(X_tensor)
        loss = loss_fn(pred, y_tensor)
        loss.backward()
        optimizer.step()

    return model, y_mean, y_std

def cnn_predict(model, X, y_mean, y_std):
    model.eval()
    with torch.no_grad():
        pred_scaled = model(torch.tensor(X, dtype=torch.float32)).numpy()
    return pred_scaled * y_std + y_mean

def cnn_gradient(model, x):
    model.eval()
    x_tensor = torch.tensor(x.reshape(1, -1), dtype=torch.float32, requires_grad=True)
    pred = model(x_tensor)
    pred.backward()
    return x_tensor.grad.detach().numpy().ravel()

def cnn_feature_importance(model, X):
    grads = []
    for x in X:
        grads.append(np.abs(cnn_gradient(model, x)))
    importance = np.mean(grads, axis=0)
    if importance.sum() > 0:
        importance = importance / importance.sum()
    return importance

# ============================================================
# Main Loop
# ============================================================

results = []
importance_rows = []
cluster_rows = []
decision_rows = []

for func_id in range(1, 9):

    print("\n" + "=" * 60)
    print(f"Function {func_id} - Week 11")
    print("=" * 60)

    X = np.load(f"function{func_id}_inputs.npy")
    y = np.load(f"function{func_id}_outputs.npy")

    for r in range(len(historical_inputs)):
        point = np.array(historical_inputs[r][func_id - 1], dtype=float)
        output = float(historical_outputs[r][func_id - 1])

        if point.shape[0] != X.shape[1]:
            raise ValueError(f"Function {func_id}: dimension mismatch.")

        if not pair_exists(X, y, point, output):
            X = np.vstack([X, point])
            y = np.append(y, output)

    dim = X.shape[1]

    best_idx = np.argmax(y)
    best_input = X[best_idx]
    best_output = y[best_idx]

    latest_output = historical_outputs[-1][func_id - 1]
    previous_best = np.max(y[:-1])
    latest_improved = latest_output >= previous_best

    recent_outputs = y[-5:]
    plateau_detected = (
        np.max(recent_outputs) - np.min(recent_outputs)
    ) < 0.05 * (np.std(y) + 1e-8)

    beta, noise, temperature = adaptive_strategy_params(
        dim, latest_improved, plateau_detected
    )

    print("Dataset size:", X.shape)
    print("Best output:", best_output)
    print("Best input:", best_input)

    # ========================================================
    # Clustering Layer
    # ========================================================

    n_clusters = min(4, max(2, len(X) // 5))

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=10
    )

    cluster_ids = kmeans.fit_predict(X)
    centroids = kmeans.cluster_centers_

    cluster_scores = []
    cluster_sizes = []

    for c in range(n_clusters):
        idx = np.where(cluster_ids == c)[0]
        cluster_sizes.append(len(idx))
        cluster_scores.append(float(np.mean(y[idx])))

    best_cluster = int(np.argmax(cluster_scores))
    best_cluster_centroid = centroids[best_cluster]

    print("Best cluster:", best_cluster)
    print("Best cluster mean:", cluster_scores[best_cluster])

    # ========================================================
    # GP Model
    # ========================================================

    kernel = (
        ConstantKernel(1.0)
        * RBF(length_scale=np.ones(dim))
        + WhiteKernel(noise_level=1e-5)
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        alpha=1e-6,
        random_state=42
    )
    gp.fit(X, y)

    # ========================================================
    # SVM Model
    # ========================================================

    threshold = np.quantile(y, 0.70)
    labels = (y >= threshold).astype(int)

    if len(np.unique(labels)) > 1:
        svm = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(
                kernel="rbf",
                probability=True,
                C=1.0,
                gamma="scale",
                random_state=42
            ))
        ])
        svm.fit(X, labels)
        use_svm = True
    else:
        svm = None
        use_svm = False

    # ========================================================
    # MLP Model
    # ========================================================

    mlp = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(128, 64, 32),
            activation="relu",
            alpha=0.0005,
            learning_rate_init=0.001,
            max_iter=1200,
            random_state=42
        ))
    ])
    mlp.fit(X, y)

    # ========================================================
    # CNN Model
    # ========================================================

    if dim <= 3:
        cnn_channels, cnn_epochs, cnn_lr = 8, 550, 0.010
    elif dim <= 5:
        cnn_channels, cnn_epochs, cnn_lr = 16, 700, 0.008
    else:
        cnn_channels, cnn_epochs, cnn_lr = 24, 850, 0.006

    cnn_model, y_mean, y_std = train_cnn(
        X,
        y,
        channels=cnn_channels,
        epochs=cnn_epochs,
        lr=cnn_lr
    )

    cnn_importance = cnn_feature_importance(cnn_model, X)

    imp_row = {"Function": func_id}
    for j in range(dim):
        imp_row[f"x{j+1}_importance"] = cnn_importance[j]
    importance_rows.append(imp_row)

    # ========================================================
    # Candidate Generation
    # ========================================================

    if dim <= 3:
        n_global, n_local, n_cluster = 9000, 3000, 2500
        grad_lr, grad_steps = 0.035, 30
    elif dim <= 5:
        n_global, n_local, n_cluster = 12000, 3500, 3000
        grad_lr, grad_steps = 0.045, 35
    else:
        n_global, n_local, n_cluster = 16000, 4000, 3500
        grad_lr, grad_steps = 0.055, 40

    global_candidates = np.random.uniform(0, 1, size=(n_global, dim))

    local_candidates = np.clip(
        best_input + np.random.normal(0, noise, size=(n_local, dim)),
        0,
        1
    )

    cluster_candidates = np.clip(
        best_cluster_centroid + np.random.normal(0, noise, size=(n_cluster, dim)),
        0,
        1
    )

    top_k = min(7, len(y))
    top_indices = np.argsort(y)[-top_k:]
    gradient_candidates = []

    for idx in top_indices:
        x = X[idx].copy()

        for _ in range(grad_steps):
            grad = cnn_gradient(cnn_model, x)
            grad = grad * (0.5 + cnn_importance)

            norm = np.linalg.norm(grad)
            if norm > 1e-12:
                grad = grad / norm

            x = np.clip(x + grad_lr * grad, 0, 1)

        gradient_candidates.append(x)

    gradient_candidates = np.array(gradient_candidates)

    candidates = np.vstack([
        global_candidates,
        local_candidates,
        cluster_candidates,
        gradient_candidates
    ])

    # ========================================================
    # Scoring
    # ========================================================

    mu, sigma = gp.predict(candidates, return_std=True)
    ucb = mu + beta * sigma

    mlp_pred = mlp.predict(candidates)
    cnn_pred = cnn_predict(cnn_model, candidates, y_mean, y_std)

    if use_svm:
        svm_prob = svm.predict_proba(candidates)[:, 1]
    else:
        svm_prob = np.ones(len(candidates))

    grad_strength = np.array([
        np.linalg.norm(cnn_gradient(cnn_model, c))
        for c in candidates
    ])

    # Attention-style score: closeness to recent points
    recent_points = X[-5:]
    attention_dist = np.array([
        np.mean([np.linalg.norm(c - rp) for rp in recent_points])
        for c in candidates
    ])
    attention_score = normalize_score(1.0 / (attention_dist + 1e-6))

    # Diversity bonus: distance from all previous points
    diversity_dist = np.array([
        np.min([np.linalg.norm(c - old) for old in X])
        for c in candidates
    ])
    diversity_bonus = normalize_score(diversity_dist)

    # Cluster proximity to best-performing cluster centroid
    cluster_dist = pairwise_distances(
        candidates,
        best_cluster_centroid.reshape(1, -1)
    ).flatten()

    cluster_score = normalize_score(np.exp(-cluster_dist))

    # Boundary tightening cue:
    # points close to boundary between best cluster and next nearest cluster
    all_centroid_distances = pairwise_distances(candidates, centroids)

    sorted_distances = np.sort(all_centroid_distances, axis=1)

    boundary_distance = sorted_distances[:, 1] - sorted_distances[:, 0]

    boundary_score = normalize_score(-boundary_distance)

    ucb_s = normalize_score(ucb)
    mlp_s = normalize_score(mlp_pred)
    cnn_s = normalize_score(cnn_pred)
    grad_s = normalize_score(grad_strength)

    base_score = (
        0.22 * ucb_s
        + 0.16 * mlp_s
        + 0.16 * cnn_s
        + 0.10 * svm_prob
        + 0.10 * grad_s
        + 0.08 * attention_score
        + 0.08 * diversity_bonus
        + 0.07 * cluster_score
        + 0.03 * boundary_score
    )

    final_score = base_score / max(temperature, 1e-8)

    best_candidate_idx = np.argmax(final_score)

    new_point = np.clip(candidates[best_candidate_idx], 0, 1)

    query = "-".join([f"{v:.6f}" for v in new_point])

    selected_cluster_distance = float(cluster_dist[best_candidate_idx])
    selected_boundary_distance = float(boundary_distance[best_candidate_idx])

    reasoning = cluster_reasoning(
        best_cluster,
        cluster_scores[best_cluster],
        selected_boundary_distance,
        selected_cluster_distance
    )

    print("Week 11 Submission:", query)
    print("Cluster reasoning:", reasoning)

    # ========================================================
    # Save Logs
    # ========================================================

    results.append({
        "Function": func_id,
        "Dimension": dim,
        "Best_Output": float(best_output),
        "Week11_Query": query
    })

    cluster_rows.append({
        "Function": func_id,
        "Number_of_Clusters": int(n_clusters),
        "Best_Cluster": int(best_cluster),
        "Best_Cluster_Mean_Output": float(cluster_scores[best_cluster]),
        "Best_Cluster_Size": int(cluster_sizes[best_cluster]),
        "Selected_Cluster_Distance": selected_cluster_distance,
        "Selected_Boundary_Distance": selected_boundary_distance,
        "Reasoning": reasoning
    })

    decision_rows.append({
        "Function": func_id,
        "Dimension": dim,
        "Best_Output": float(best_output),
        "Latest_Output": float(latest_output),
        "Latest_Improved": bool(latest_improved),
        "Plateau_Detected": bool(plateau_detected),
        "Beta": float(beta),
        "Noise": float(noise),
        "Temperature": float(temperature),
        "Week11_Query": query,
        "Selected_UCB_Component": float(ucb_s[best_candidate_idx]),
        "Selected_MLP_Component": float(mlp_s[best_candidate_idx]),
        "Selected_CNN_Component": float(cnn_s[best_candidate_idx]),
        "Selected_SVM_Probability": float(svm_prob[best_candidate_idx]),
        "Selected_Gradient_Component": float(grad_s[best_candidate_idx]),
        "Selected_Attention_Component": float(attention_score[best_candidate_idx]),
        "Selected_Diversity_Component": float(diversity_bonus[best_candidate_idx]),
        "Selected_Cluster_Component": float(cluster_score[best_candidate_idx]),
        "Selected_Boundary_Component": float(boundary_score[best_candidate_idx]),
        "Reasoning": reasoning
    })

# ============================================================
# Save Results
# ============================================================

pd.DataFrame(results).to_csv("Week11_Submissions.csv", index=False)
pd.DataFrame(importance_rows).to_csv("Week11_CNN_Feature_Importance.csv", index=False)
pd.DataFrame(cluster_rows).to_csv("Week11_Cluster_Log.csv", index=False)
pd.DataFrame(decision_rows).to_csv("Week11_Decision_Log.csv", index=False)

print("\nDone ✅")
print("Saved: Week11_Submissions.csv")
print("Saved: Week11_CNN_Feature_Importance.csv")
print("Saved: Week11_Cluster_Log.csv")
print("Saved: Week11_Decision_Log.csv")


Function 1 - Week 11
Dataset size: (19, 2)
Best output: 7.710875114502849e-16
Best input: [0.73102363 0.73299988]
Best cluster: 2
Best cluster mean: 2.6453540790908152e-80
Week 11 Submission: 0.467104-0.834974
Cluster reasoning: Query targets cluster 2, which has a strong mean output of 0.000000. The candidate is guided by centroid proximity (0.299347) and boundary distance (0.011683), balancing local cluster refinement with controlled exploration.

Function 2 - Week 11
Dataset size: (20, 2)
Best output: 0.6219708025012802
Best input: [0.700273 1.      ]
Best cluster: 1
Best cluster mean: 0.38550074582199656
Week 11 Submission: 0.747288-0.926554
Cluster reasoning: Query targets cluster 1, which has a strong mean output of 0.385501. The candidate is guided by centroid proximity (0.058190) and boundary distance (0.665925), balancing local cluster refinement with controlled exploration.

Function 3 - Week 11
Dataset size: (25, 3)
Best output: -0.011085004696869778
Best input: [0.645025 0